In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
from pybaseball import playerid_lookup
from pybaseball import statcast
import ipywidgets as widgets
from IPython.display import display, clear_output



df = statcast("2026-03-25", "2026-09-18")




def classify_swing_take(description):

    swing_descriptions = [
        "foul",
        "hit_into_play",
        "swinging_strike",
        "foul_tip",
        "swinging_strike_blocked",
        "foul_bunt",
        "missed_bunt",
        "bunt_foul_tip"
    ]

    take_descriptions = [
        "ball",
        "called_strike",
        "blocked_ball",
        "automatic_ball",
        "hit_by_pitch",
        "automatic_strike",
        "pitchout"
    ]

    if description in swing_descriptions:
        return "Swing"

    elif description in take_descriptions:
        return "Take"

    else:
        return "Unknown"


df["swing_take"] = df["description"].apply(classify_swing_take)


def get_batter_location(row):

    zone = row["zone"]
    stand = row["stand"]

    if pd.isna(zone):
        return "Unknown"

    zone = int(zone)

    # Vertical location
    if zone in [1, 2, 3, 11, 12]:
        vertical = "High"
    elif zone in [4, 5, 6]:
        vertical = "Middle"
    elif zone in [7, 8, 9, 13, 14]:
        vertical = "Low"
    else:
        vertical = "Other"

    # Horizontal location
    if zone in [1, 4, 7]:
        horizontal = "Inside" if stand == "R" else "Away"

    elif zone in [2, 5, 8]:
        horizontal = "Middle"

    elif zone in [3, 6, 9]:
        horizontal = "Away" if stand == "R" else "Inside"

    elif zone in [11, 13]:
        horizontal = "Inside" if stand == "R" else "Away"

    elif zone in [12, 14]:
        horizontal = "Away" if stand == "R" else "Inside"

    else:
        horizontal = "Other"

    return horizontal + " " + vertical


df["batter_location"] = df.apply(get_batter_location, axis=1)



RED = "#C62828"
BLUE = "#1565C0"
DARK = "#222222"
GRAY = "#777777"
LIGHT_GRAY = "#E5E5E5"
WHITE = "#FFFFFF"

RV_CMAP = "RdBu_r"
MIN_COLOR_SAMPLE = 25






analysis_columns = [
    "player_name",
    "batter",
    "pitch_name",
    "pitch_type",
    "zone",
    "stand",
    "batter_location",
    "swing_take",
    "delta_run_exp",
    "plate_x",
    "plate_z",
    "balls",
    "strikes",
    "outs_when_up",
    "on_1b",
    "on_2b",
    "on_3b",
    "inning",
    "inning_topbot",
    "bat_score",
    "fld_score"
]

analysis_df = df[analysis_columns].copy()

analysis_df = analysis_df.dropna(
    subset=[
        "batter",
        "pitch_name",
        "zone",
        "swing_take",
        "delta_run_exp"
    ]
)

analysis_df["batter"] = pd.to_numeric(
    analysis_df["batter"],
    errors="coerce"
)

analysis_df["zone"] = pd.to_numeric(
    analysis_df["zone"],
    errors="coerce"
)

analysis_df["delta_run_exp"] = pd.to_numeric(
    analysis_df["delta_run_exp"],
    errors="coerce"
)

analysis_df["balls"] = pd.to_numeric(
    analysis_df["balls"],
    errors="coerce"
)

analysis_df["strikes"] = pd.to_numeric(
    analysis_df["strikes"],
    errors="coerce"
)

analysis_df = analysis_df.dropna(
    subset=[
        "batter",
        "zone",
        "delta_run_exp",
        "balls",
        "strikes"
    ]
)

analysis_df["zone"] = (
    analysis_df["zone"]
    .astype(int)
)


def classify_count_group(row):

    balls = row["balls"]
    strikes = row["strikes"]

    if strikes == 2:
        return "Two Strikes"

    elif balls > strikes:
        return "Hitter's Count"

    elif strikes > balls:
        return "Pitcher's Count"

    else:
        return "Even Count"


analysis_df["count_group"] = analysis_df.apply(
    classify_count_group,
    axis=1
)

count_order = [
    "Hitter's Count",
    "Even Count",
    "Pitcher's Count",
    "Two Strikes"
]


league_zone = (
    analysis_df
    .groupby(
        ["batter", "zone"]
    )
    .agg(
        pitches=("delta_run_exp", "size"),
        total_rv=("delta_run_exp", "sum")
    )
    .reset_index()
)

league_zone["rv_per_100"] = (
    league_zone["total_rv"]
    / league_zone["pitches"]
    * 100
)

league_zone_for_color = league_zone[
    league_zone["pitches"] >= MIN_COLOR_SAMPLE
].copy()

if len(league_zone_for_color) > 0:

    rv_low = (
        league_zone_for_color["rv_per_100"]
        .quantile(0.05)
    )

    rv_high = (
        league_zone_for_color["rv_per_100"]
        .quantile(0.95)
    )

else:

    rv_low = -5
    rv_high = 5

rv_limit = max(
    abs(rv_low),
    abs(rv_high),
    1
)

RV_NORM = TwoSlopeNorm(
    vmin=-rv_limit,
    vcenter=0,
    vmax=rv_limit
)


league_count = (
    analysis_df
    .groupby(
        ["batter", "count_group"]
    )
    .agg(
        pitches=("delta_run_exp", "size"),
        total_rv=("delta_run_exp", "sum")
    )
    .reset_index()
)

league_count["rv_per_100"] = (
    league_count["total_rv"]
    / league_count["pitches"]
    * 100
)

league_count_for_color = league_count[
    league_count["pitches"] >= MIN_COLOR_SAMPLE
].copy()

if len(league_count_for_color) > 0:

    count_rv_low = (
        league_count_for_color["rv_per_100"]
        .quantile(0.05)
    )

    count_rv_high = (
        league_count_for_color["rv_per_100"]
        .quantile(0.95)
    )

else:

    count_rv_low = -5
    count_rv_high = 5

count_rv_limit = max(
    abs(count_rv_low),
    abs(count_rv_high),
    1
)

COUNT_RV_NORM = TwoSlopeNorm(
    vmin=-count_rv_limit,
    vcenter=0,
    vmax=count_rv_limit
)


def find_hitter(first_name, last_name):

    lookup = playerid_lookup(
        last_name,
        first_name
    )

    if lookup.empty:
        return None

    lookup = lookup.dropna(
        subset=["key_mlbam"]
    )

    if lookup.empty:
        return None

    exact = lookup[
        (
            lookup["name_first"]
            .astype(str)
            .str.lower()
            == first_name.lower()
        )
        &
        (
            lookup["name_last"]
            .astype(str)
            .str.lower()
            == last_name.lower()
        )
    ]

    if not exact.empty:

        return int(
            exact.iloc[0]["key_mlbam"]
        )

    return int(
        lookup.iloc[0]["key_mlbam"]
    )


ZONE_POSITIONS = {
    1: (0, 2),
    2: (1, 2),
    3: (2, 2),
    4: (0, 1),
    5: (1, 1),
    6: (2, 1),
    7: (0, 0),
    8: (1, 0),
    9: (2, 0),
    11: (-1, 2),
    12: (3, 2),
    13: (-1, 0),
    14: (3, 0)
}


def draw_rv_zone_grid(
    ax,
    zone_values,
    norm,
    cmap
):

    for zone, (x, y) in ZONE_POSITIONS.items():

        value = zone_values.get(
            zone,
            np.nan
        )

        if pd.isna(value):

            face_color = "#D9D9D9"

        else:

            face_color = cmap(
                norm(value)
            )

        rect = Rectangle(
            (x, y),
            1,
            1,
            facecolor=face_color,
            edgecolor=WHITE,
            linewidth=1.5
        )

        ax.add_patch(rect)

        ax.text(
            x + 0.10,
            y + 0.86,
            str(zone),
            fontsize=8,
            fontweight="bold",
            color=DARK,
            ha="left",
            va="top"
        )

        if not pd.isna(value):

            text_color = (
                WHITE
                if abs(value) > rv_limit * 0.45
                else DARK
            )

            ax.text(
                x + 0.5,
                y + 0.47,
                f"{value:+.2f}",
                fontsize=8,
                fontweight="bold",
                color=text_color,
                ha="center",
                va="center"
            )

    strike_zone = Rectangle(
        (0, 0),
        3,
        3,
        fill=False,
        edgecolor=DARK,
        linewidth=2
    )

    ax.add_patch(strike_zone)

    ax.set_xlim(-1, 4)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")


def draw_swing_zone_grid(
    ax,
    zone_values,
    norm,
    cmap,
    overall_swing_rate
):

    for zone, (x, y) in ZONE_POSITIONS.items():

        value = zone_values.get(
            zone,
            np.nan
        )

        if pd.isna(value):

            face_color = "#D9D9D9"

        else:

            face_color = cmap(
                norm(value)
            )

        rect = Rectangle(
            (x, y),
            1,
            1,
            facecolor=face_color,
            edgecolor=WHITE,
            linewidth=1.5
        )

        ax.add_patch(rect)

        ax.text(
            x + 0.10,
            y + 0.86,
            str(zone),
            fontsize=8,
            fontweight="bold",
            color=DARK,
            ha="left",
            va="top"
        )

        if not pd.isna(value):

            difference = (
                value
                - overall_swing_rate
            )

            text_color = (
                WHITE
                if abs(difference) > 15
                else DARK
            )

            ax.text(
                x + 0.5,
                y + 0.47,
                f"{value:.1f}%",
                fontsize=8,
                fontweight="bold",
                color=text_color,
                ha="center",
                va="center"
            )

    strike_zone = Rectangle(
        (0, 0),
        3,
        3,
        fill=False,
        edgecolor=DARK,
        linewidth=2
    )

    ax.add_patch(strike_zone)

    ax.set_xlim(-1, 4)
    ax.set_ylim(0, 3)
    ax.set_aspect("equal")
    ax.axis("off")


def create_hitter_report(
    first_name,
    last_name,
    pitch_filter="All Pitches"
):

    batter_id = find_hitter(
        first_name,
        last_name
    )

    if batter_id is None:

        print(
            f"Could not find {first_name} {last_name}."
        )

        return None

    display_name = (
        first_name.strip().title()
        + " "
        + last_name.strip().title()
    )

    hitter_df = analysis_df[
        analysis_df["batter"] == batter_id
    ].copy()

    if hitter_df.empty:

        print(
            f"No Statcast data found for "
            f"{display_name}."
        )

        return None

    if pitch_filter != "All Pitches":

        hitter_df = hitter_df[
            hitter_df["pitch_name"]
            == pitch_filter
        ].copy()

        if hitter_df.empty:

            print(
                f"No data found for "
                f"{display_name} on {pitch_filter}."
            )

            return None

    total_pitches = len(
        hitter_df
    )

    total_swings = (
        hitter_df["swing_take"]
        == "Swing"
    ).sum()

    total_takes = (
        hitter_df["swing_take"]
        == "Take"
    ).sum()

    swing_rate = (
        total_swings
        / total_pitches
        * 100
    )

    total_rv = hitter_df[
        "delta_run_exp"
    ].sum()

    rv_per_100 = (
        total_rv
        / total_pitches
        * 100
    )

    fig = plt.figure(
        figsize=(17, 18),
        facecolor="white"
    )

    gs = fig.add_gridspec(
        nrows=24,
        ncols=12,
        left=0.065,
        right=0.94,
        top=0.91,
        bottom=0.055,
        hspace=2.0,
        wspace=1.15
    )

    fig.text(
        0.065,
        0.965,
        display_name,
        fontsize=26,
        fontweight="bold",
        color=DARK,
        ha="left",
        va="top"
    )

    fig.text(
        0.065,
        0.94,
        "2026 Statcast Hitter Report",
        fontsize=11,
        fontweight="bold",
        color=RED,
        ha="left",
        va="top"
    )

    if pitch_filter != "All Pitches":

        fig.text(
            0.94,
            0.965,
            pitch_filter,
            fontsize=12,
            fontweight="bold",
            color=GRAY,
            ha="right",
            va="top"
        )

    metrics = [
        ("PITCHES", f"{total_pitches:,}"),
        ("SWINGS", f"{total_swings:,}"),
        ("TAKES", f"{total_takes:,}"),
        ("SWING %", f"{swing_rate:.1f}%"),
        ("TOTAL RV", f"{total_rv:+.2f}"),
        ("RV / 100", f"{rv_per_100:+.2f}")
    ]

    metric_x = np.linspace(
        0.065,
        0.72,
        len(metrics)
    )

    for x, (label, value) in zip(
        metric_x,
        metrics
    ):

        fig.text(
            x,
            0.895,
            label,
            fontsize=7,
            fontweight="bold",
            color=BLUE,
            ha="left"
        )

        fig.text(
            x,
            0.872,
            value,
            fontsize=16,
            fontweight="bold",
            color=DARK,
            ha="left"
        )

    ax1 = fig.add_subplot(
        gs[2:8, 0:6]
    )

    pitch_usage = (
        hitter_df
        .groupby("pitch_name")
        .agg(
            pitches=("pitch_name", "size"),
            swings=(
                "swing_take",
                lambda x:
                (x == "Swing").sum()
            ),
            takes=(
                "swing_take",
                lambda x:
                (x == "Take").sum()
            )
        )
        .reset_index()
    )

    pitch_usage["usage_pct"] = (
        pitch_usage["pitches"]
        / total_pitches
        * 100
    )

    pitch_usage["swing_pct"] = (
        pitch_usage["swings"]
        / pitch_usage["pitches"]
        * 100
    )

    pitch_usage["take_pct"] = (
        pitch_usage["takes"]
        / pitch_usage["pitches"]
        * 100
    )

    pitch_usage = pitch_usage.sort_values(
        "usage_pct",
        ascending=True
    )

    y = np.arange(
        len(pitch_usage)
    )

    swing_width = (
        pitch_usage["usage_pct"]
        * pitch_usage["swing_pct"]
        / 100
    )

    take_width = (
        pitch_usage["usage_pct"]
        * pitch_usage["take_pct"]
        / 100
    )

    ax1.barh(
        y,
        swing_width,
        color=RED,
        height=0.62,
        label="Swing"
    )

    ax1.barh(
        y,
        take_width,
        left=swing_width,
        color=BLUE,
        height=0.62,
        label="Take"
    )

    for i, usage in enumerate(
        pitch_usage["usage_pct"]
    ):

        ax1.text(
            usage + 0.35,
            i,
            f"{usage:.1f}%",
            va="center",
            ha="left",
            fontsize=8,
            color=DARK,
            fontweight="bold"
        )

    for i, (
        swing_width_value,
        swing_pct_value
    ) in enumerate(
        zip(
            swing_width,
            pitch_usage["swing_pct"]
        )
    ):

        if swing_width_value >= 1.8:

            ax1.text(
                swing_width_value / 2,
                i,
                f"{swing_pct_value:.0f}%",
                va="center",
                ha="center",
                fontsize=7,
                color=WHITE,
                fontweight="bold"
            )

    for i, (
        swing_width_value,
        take_width_value,
        take_pct_value
    ) in enumerate(
        zip(
            swing_width,
            take_width,
            pitch_usage["take_pct"]
        )
    ):

        if take_width_value >= 1.8:

            ax1.text(
                swing_width_value
                + take_width_value / 2,
                i,
                f"{take_pct_value:.0f}%",
                va="center",
                ha="center",
                fontsize=7,
                color=WHITE,
                fontweight="bold"
            )

    ax1.set_yticks(y)

    ax1.set_yticklabels(
        pitch_usage["pitch_name"],
        fontsize=8
    )

    ax1.set_xlabel(
        "Share of all pitches (%)",
        fontsize=9,
        labelpad=7
    )

    ax1.set_title(
        "Pitch Usage & Swing / Take",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=14,
        color=DARK
    )

    ax1.legend(
        loc="lower right",
        frameon=False,
        fontsize=8
    )

    ax1.grid(
        axis="x",
        alpha=0.2
    )

    ax1.spines[
        ["top", "right", "left"]
    ].set_visible(False)

    ax1.tick_params(
        axis="x",
        labelsize=8
    )

    ax2 = fig.add_subplot(
        gs[2:8, 6:12]
    )

    count_summary = (
        hitter_df
        .groupby("count_group")
        .agg(
            pitches=("delta_run_exp", "size"),
            total_rv=("delta_run_exp", "sum")
        )
        .reindex(count_order)
        .reset_index()
    )

    count_summary["rv_per_100"] = (
        count_summary["total_rv"]
        / count_summary["pitches"]
        * 100
    )

    count_summary = count_summary[
        count_summary["pitches"] > 0
    ].copy()

    count_colors = [
        plt.get_cmap(RV_CMAP)(
            COUNT_RV_NORM(value)
        )
        for value in
        count_summary["rv_per_100"]
    ]

    bars = ax2.bar(
        count_summary["count_group"],
        count_summary["rv_per_100"],
        color=count_colors,
        width=0.62
    )

    ax2.axhline(
        0,
        color=GRAY,
        linewidth=0.8
    )

    for bar, value in zip(
        bars,
        count_summary["rv_per_100"]
    ):

        offset = max(
            count_rv_limit * 0.025,
            0.08
        )

        if value >= 0:

            y_position = value + offset
            va = "bottom"

        else:

            y_position = value - offset
            va = "top"

        ax2.text(
            bar.get_x()
            + bar.get_width() / 2,
            y_position,
            f"{value:+.2f}",
            ha="center",
            va=va,
            fontsize=8,
            fontweight="bold",
            color=DARK
        )

    ax2.set_ylabel(
        "RV / 100 Pitches",
        fontsize=9,
        labelpad=7
    )

    ax2.set_title(
        "Run Value by Count",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=14,
        color=DARK
    )

    ax2.tick_params(
        axis="x",
        labelrotation=15,
        labelsize=8
    )

    ax2.grid(
        axis="y",
        alpha=0.2
    )

    ax2.spines[
        ["top", "right"]
    ].set_visible(False)

    ax3 = fig.add_subplot(
        gs[9:15, 0:6]
    )

    zone_summary = (
        hitter_df
        .groupby("zone")
        .agg(
            pitches=("delta_run_exp", "size"),
            total_rv=("delta_run_exp", "sum")
        )
    )

    zone_summary["rv_per_100"] = (
        zone_summary["total_rv"]
        / zone_summary["pitches"]
        * 100
    )

    zone_values = (
        zone_summary["rv_per_100"]
        .to_dict()
    )

    draw_rv_zone_grid(
        ax=ax3,
        zone_values=zone_values,
        norm=RV_NORM,
        cmap=plt.get_cmap(RV_CMAP)
    )

    ax3.set_title(
        "Run Value by Pitch Location",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=14,
        color=DARK
    )

    ax4 = fig.add_subplot(
        gs[9:15, 6:12]
    )

    swing_zone_summary = (
        hitter_df
        .groupby("zone")
        .agg(
            pitches=("swing_take", "size"),
            swings=(
                "swing_take",
                lambda x:
                (x == "Swing").sum()
            )
        )
    )

    swing_zone_summary["swing_rate"] = (
        swing_zone_summary["swings"]
        / swing_zone_summary["pitches"]
        * 100
    )

    swing_zone_values = (
        swing_zone_summary["swing_rate"]
        .to_dict()
    )

    overall_swing_rate = swing_rate

    swing_min = max(
        0,
        overall_swing_rate - 35
    )

    swing_max = min(
        100,
        overall_swing_rate + 35
    )

    if swing_min == swing_max:

        swing_min = 0
        swing_max = 100

    swing_norm = TwoSlopeNorm(
        vmin=swing_min,
        vcenter=overall_swing_rate,
        vmax=swing_max
    )

    draw_swing_zone_grid(
        ax=ax4,
        zone_values=swing_zone_values,
        norm=swing_norm,
        cmap=plt.get_cmap(RV_CMAP),
        overall_swing_rate=overall_swing_rate
    )

    ax4.set_title(
        "Swing Rate by Pitch Location",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=14,
        color=DARK
    )

    fig.text(
        0.065,
        0.345,
        "Pitch location is shown from the catcher's perspective. For a right-handed hitter, 1, 4, and 7 represent the inside portion of the zone.",
        fontsize=7.5,
        color=GRAY,
        ha="left"
    )

    cbar_ax = fig.add_axes(
        [
            0.952,
            0.355,
            0.012,
            0.22
        ]
    )

    cbar_ax.grid(False)

    sm = ScalarMappable(
        norm=RV_NORM,
        cmap=RV_CMAP
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cbar_ax
    )

    cbar.set_label(
        "RV / 100 Pitches",
        fontsize=8
    )

    cbar.ax.tick_params(
        labelsize=7
    )

    table_df = (
        hitter_df
        .groupby(
            [
                "pitch_name",
                "batter_location"
            ]
        )
        .agg(
            pitches=("delta_run_exp", "size"),
            swings=(
                "swing_take",
                lambda x:
                (x == "Swing").sum()
            ),
            total_rv=("delta_run_exp", "sum")
        )
        .reset_index()
    )

    table_df["usage_pct"] = (
        table_df["pitches"]
        / total_pitches
        * 100
    )

    table_df["swing_pct"] = (
        table_df["swings"]
        / table_df["pitches"]
        * 100
    )

    table_df["rv_per_100"] = (
        table_df["total_rv"]
        / table_df["pitches"]
        * 100
    )

    table_df = table_df[
        table_df["pitches"] >= 5
    ].copy()

    table_df = table_df.sort_values(
        [
            "pitches",
            "pitch_name"
        ],
        ascending=[
            False,
            True
        ]
    )

    ax5 = fig.add_subplot(
        gs[16:24, 0:12]
    )

    ax5.axis("off")

    ax5.set_title(
        "Pitch Type × Location",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=12,
        color=DARK
    )

    display_table = table_df[
        [
            "pitch_name",
            "batter_location",
            "pitches",
            "usage_pct",
            "swing_pct",
            "rv_per_100"
        ]
    ].head(24).copy()

    rv_values_for_color = (
        display_table["rv_per_100"]
        .copy()
    )

    display_table["usage_pct"] = (
        display_table["usage_pct"]
        .map(
            lambda x:
            f"{x:.1f}%"
        )
    )

    display_table["swing_pct"] = (
        display_table["swing_pct"]
        .map(
            lambda x:
            f"{x:.1f}%"
        )
    )

    display_table["rv_per_100"] = (
        display_table["rv_per_100"]
        .map(
            lambda x:
            f"{x:+.2f}"
        )
    )

    display_table.columns = [
        "Pitch",
        "Location",
        "Pitches",
        "Usage",
        "Swing %",
        "RV / 100"
    ]

    table = ax5.table(
        cellText=display_table.values,
        colLabels=display_table.columns,
        cellLoc="center",
        colLoc="center",
        loc="upper center",
        bbox=[
            0,
            0.015,
            1,
            0.88
        ]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)

    for col in range(
        len(display_table.columns)
    ):

        cell = table[
            0,
            col
        ]

        cell.set_facecolor(
            BLUE
        )

        cell.set_text_props(
            color=WHITE,
            weight="bold"
        )

        cell.set_edgecolor(
            WHITE
        )

    for row in range(
        1,
        len(display_table) + 1
    ):

        for col in range(
            len(display_table.columns)
        ):

            cell = table[
                row,
                col
            ]

            cell.set_edgecolor(
                LIGHT_GRAY
            )

            cell.set_linewidth(
                0.5
            )

            if row % 2 == 0:

                cell.set_facecolor(
                    "#F7F7F7"
                )

            else:

                cell.set_facecolor(
                    WHITE
                )

        rv_value = (
            rv_values_for_color.iloc[
                row - 1
            ]
        )

        rv_cell = table[
            row,
            5
        ]

        if rv_value > 0:

            rv_cell.set_text_props(
                color=RED,
                weight="bold"
            )

        elif rv_value < 0:

            rv_cell.set_text_props(
                color=BLUE,
                weight="bold"
            )

    column_widths = [
        0.25,
        0.25,
        0.12,
        0.12,
        0.13,
        0.13
    ]

    for col, width in enumerate(
        column_widths
    ):

        for row in range(
            len(display_table) + 1
        ):

            table[
                row,
                col
            ].set_width(
                width
            )

    fig.text(
        0.065,
        0.018,
        "Run value is pitch-level change in run expectancy. Red indicates positive RV and blue indicates negative RV.",
        fontsize=7,
        color=GRAY,
        ha="left"
    )

    plt.show()

    return {
        "hitter_data": hitter_df,
        "pitch_usage": pitch_usage,
        "count_summary": count_summary,
        "zone_summary": zone_summary,
        "swing_zone_summary": swing_zone_summary,
        "pitch_location": table_df
    }


player_input = widgets.Text(
    value="Juan Soto",
    description="Hitter:",
    placeholder="First Last",
    layout=widgets.Layout(
        width="300px"
    )
)

pitch_options = [
    "All Pitches"
] + sorted(
    analysis_df["pitch_name"]
    .dropna()
    .unique()
    .tolist()
)

pitch_input = widgets.Dropdown(
    options=pitch_options,
    value="All Pitches",
    description="Pitch:",
    layout=widgets.Layout(
        width="350px"
    )
)

generate_button = widgets.Button(
    description="Generate Report",
    button_style="primary",
    layout=widgets.Layout(
        width="160px",
        height="35px"
    )
)

output = widgets.Output()


def generate_report(button):

    with output:

        clear_output(
            wait=True
        )

        name_parts = (
            player_input.value
            .strip()
            .split()
        )

        if len(name_parts) < 2:

            print(
                "Please enter a first and last name."
            )

            return

        first_name = name_parts[0]

        last_name = " ".join(
            name_parts[1:]
        )

        create_hitter_report(
            first_name,
            last_name,
            pitch_input.value
        )


generate_button.on_click(
    generate_report
)

display(
    widgets.HBox(
        [
            player_input,
            pitch_input,
            generate_button
        ]
    )
)

display(output)

This is a large query, it may take a moment to complete


100%|████████████████████████████████████████████████████████████████████████████████| 176/176 [00:44<00:00,  4.00it/s]
C:\anaconda\lib\site-packages\pybaseball\statcast.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_data = pd.concat(dataframe_list, axis=0).convert_dtypes(convert_string=False)


Output()